# Dataset 04 — scanpy vs rule-based cell typing (CosMx kidney)

[GSE282026](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE282026). Loads pre-clustered **`scanpy_cluster.h5ad`**, applies **rule-based** panels from **`celltyping_config.yaml`** (cellruler API), then compares methods.

**Kernel:** `spatialdata` — install package once: `pip install -e /mnt/scratch2/Maycon/Hackathon/SJ_BioHack_2026/KIDS26-Team18`


In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import yaml
from IPython.display import display

from cellruler.plot import SpatialCoord_plot, custom_barplot
import cellruler.clustering as clustering

try:
    import cellruler.rulebased as rulebased
except Exception:
    # Fallback: same API as cellruler.rulebased (package file currently has a syntax error).
    import types
    rulebased = types.SimpleNamespace()

    def _detection_rate(adata, genes):
        present = [g for g in genes if g in adata.var_names]
        rates = {}
        for gene in present:
            x = adata[:, gene].X
            if hasattr(x, "toarray"):
                x = x.toarray().ravel()
            else:
                x = np.asarray(x).ravel()
            rates[gene] = float((x > 0).mean())
        return pd.Series(rates).sort_values(ascending=False)

    def get_top_n_markers(adata, candidate_markers, n=4):
        final_markers = {}
        for cell_type in candidate_markers:
            rates = _detection_rate(adata, candidate_markers[cell_type])
            print(f"\nSelected {cell_type} markers:")
            display(rates.head(n))
            final_markers[cell_type] = rates.head(n).index.tolist()
        return {k: v for k, v in final_markers.items() if v}

    def _get_gene_counts(adata, genes):
        x = adata[:, genes].X
        return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

    def assign_rule_labels(adata, markers_dict):
        cell_types = list(markers_dict.keys())
        n_cells, n_panels = adata.n_obs, len(cell_types)
        scores = np.zeros((n_panels, n_cells), dtype=float)
        for p, cell_type in enumerate(cell_types):
            scores[p] = _get_gene_counts(adata, markers_dict[cell_type]).sum(axis=1)
        qualifies = scores > 0
        panel_order = np.arange(n_panels).reshape(-1, 1)
        rank = np.where(qualifies, scores + panel_order * 1e-12, -np.inf)
        best_panel_idx = rank.argmax(axis=0)
        has_hit = qualifies.any(axis=0)
        labels = np.full(n_cells, "Unassigned", dtype=object)
        labels[has_hit] = np.array(cell_types, dtype=object)[best_panel_idx[has_hit]]
        return labels

    rulebased.get_top_n_markers = get_top_n_markers
    rulebased.assign_rule_labels = assign_rule_labels
    print("Using in-notebook fallback for cellruler.rulebased (fix cellruler/rulebased.py to import package module).")

PROC_DIR = Path("/mnt/scratch2/Maycon/Hackathon/SJ_BioHack_2026/KIDS26-Team18/data/Dataset_04/processed_data")
H5AD_PATH = PROC_DIR / "scanpy_cluster.h5ad"
CONFIG_PATH = PROC_DIR / "celltyping_config.yaml"
OUTPUT_H5AD = PROC_DIR / "scanpy_cluster_cellbased.h5ad"


In [ ]:
with open(CONFIG_PATH) as f:
    CFG = yaml.safe_load(f)

RULE = CFG["rule"]
OBS_KEYS = CFG["obs_keys"]
candidate_markers = {
    panel: spec["candidates"]
    for panel, spec in CFG["rule_panels"].items()
}

adata = ad.read_h5ad(H5AD_PATH)
print(f"Loaded {adata.n_obs:,} cells from {H5AD_PATH.name}")
print("leiden:", "leiden" in adata.obs, "| cell_type_scanpy:", OBS_KEYS["cell_type_scanpy"] in adata.obs)
print("obsm spatial:", "spatial" in adata.obsm, "| X_umap:", "X_umap" in adata.obsm)


## 2. Approach i — scanpy (pre-clustered)

`scanpy_cluster.h5ad` already contains Leiden clusters and `cell_type_scanpy`. Optional: review rank-gene tables if present.


In [ ]:
scanpy_key = OBS_KEYS["cell_type_scanpy"]
leiden_key = OBS_KEYS["leiden"]

if scanpy_key not in adata.obs and "cluster_annotations" in CFG:
    adata = clustering.annotate_clusters(adata, CFG["cluster_annotations"])

if "coord_x" not in adata.obs and "spatial" in adata.obsm:
    adata.obs["coord_x"] = adata.obsm["spatial"][:, 0]
    adata.obs["coord_y"] = adata.obsm["spatial"][:, 1]

if "rank_genes_groups" in adata.uns and adata.obs[leiden_key].notna().all():
    top_n = int(CFG.get("scanpy", {}).get("rank_genes_top_n", 10))
    try:
        markers_topn = clustering.get_topn_markers(adata, top_n=top_n)
        display(markers_topn.head(20))
    except ValueError as err:
        print(f"Skipping marker table: {err}")
else:
    print("Skipping rank-gene marker table (missing rank_genes_groups or NaN leiden labels).")

sc.pl.umap(
    adata,
    color=[leiden_key, scanpy_key],
    wspace=0.4,
    title=["Leiden", "Scanpy manual"],
)
SpatialCoord_plot(
    adata,
    variable=scanpy_key,
    size=4,
    alpha=0.7,
    title="Spatial — scanpy",
)


## 3. Approach ii — rule-based (yaml panels + cellruler)

Top **`rule.top_n_markers`** genes per panel by detection rate, then `assign_rule_labels`.


In [ ]:
if "coord_x" not in adata.obs and "spatial" in adata.obsm:
    adata.obs["coord_x"] = adata.obsm["spatial"][:, 0]
    adata.obs["coord_y"] = adata.obsm["spatial"][:, 1]

n_markers = int(RULE["top_n_markers"])
final_markers = rulebased.get_top_n_markers(adata, candidate_markers, n=n_markers)

rule_key = OBS_KEYS["cell_type_rule"]
adata.obs[rule_key] = pd.Categorical(rulebased.assign_rule_labels(adata, final_markers))

print("Rule-based labels:")
display(adata.obs[rule_key].value_counts())


In [ ]:
sc.pl.umap(adata, color=rule_key, title="Rule-based")
SpatialCoord_plot(
    adata,
    variable=rule_key,
    size=4,
    alpha=0.7,
    title="Spatial — rule-based",
)

# Dotplot for first panel with markers
first_panel = next(iter(final_markers))
sc.pl.dotplot(
    adata,
    var_names=final_markers[first_panel],
    groupby=rule_key,
    standard_scale="var",
    title=f"Markers — {first_panel}",
)


In [ ]:
# Optional: save annotated object (gitignored .h5ad)
# adata.write_h5ad(OUTPUT_H5AD)


## 4. Compare scanpy (i) vs rule-based (ii)


In [ ]:
print("Comparison\n")
print("=== Counts ===")
summary = pd.DataFrame({
    scanpy_key: adata.obs[scanpy_key].value_counts(),
    rule_key: adata.obs[rule_key].value_counts(),
})
display(summary)

print("\n=== Crosstab (scanpy rows × rule cols) ===")
ct = pd.crosstab(adata.obs[scanpy_key], adata.obs[rule_key], margins=True)
display(ct)

print("\n=== Row-normalized crosstab ===")
ct_norm = pd.crosstab(
    adata.obs[scanpy_key],
    adata.obs[rule_key],
    normalize="index",
).round(3)
display(ct_norm)

strict = (adata.obs[scanpy_key].astype(str) == adata.obs[rule_key].astype(str)).mean()
print(f"\nStrict match rate: {100 * strict:.2f}%")


In [ ]:
custom_barplot(
    adata,
    var_1=rule_key,
    var_2=scanpy_key,
    plot_type="bar",
    cluster_bars=False,
    title="Rule labels within scanpy categories",
)
